# 00 — Overview (Start Here)

**Question.** What volumetric rate constraint follows from an empirical, untargeted kilonova search across eight years of ZTF observations (2018 March – 2026 March, Phases I + II + O4) paired with matched injection–recovery simulations?

Every number on this page is computed directly from the two version-controlled snapshots:
1. `snapshots/knrates_results_latest.zip` — `survey_sim` kilonova injection–recovery ensembles;
2. `../real-ztf-search/snapshots/knsearch_results_latest.zip` — archival ZTF alert search, real–bogus epoch calibration, benchmark recoveries, and archive coverage.

| Notebook | Question it answers | Manuscript |
|---|---|---|
| `01_archival_search_and_efficiency` | How does the 8-yr archival search suppress background, calibrate RB epochs, recover known fast transients, and measure $\varepsilon_{\mathrm{pipe}}$? | Paper I §§2–3 |
| `02_limits_and_timeline` | What are $\langle VT\rangle$, the 90/95/99% rate limits, distance reach, 8-yr cumulative growth, and comparison with Andreoni+2020/2021 & GWTC-4.0? | Paper I §§4.1, 5.1–5.2, Figs. 1, 3–4, Table 1 |
| `03_viewing_angle` | How strongly does binary inclination $\iota$ control detectability? | Paper I §5.3, Fig. 2 |
| `04_systematics_and_convergence` | How robust are the limits to MC noise, $z_{\max}$, ejecta shifts, AT2017gfo light-curve residuals, criteria, and broad populations? | Paper I §5.4, App. A, Fig. 5 |
| `paper_figures` | Rebuilds Paper I `paper/figures/fig1..5.{pdf,png}`, `paper/table1_master_rate_limits.{csv,tex}`, and `paper/numbers.tex` | Paper I assets |
| `companion/05_criteria_and_imposters` | How does each selection criterion trade $\langle VT\rangle$ against empirical ZTF background & extragalactic imposters? | Paper II, Fig. 6 |
| `companion/paper2_figures` | Rebuilds Paper II `paper/figures/fig6_criteria_sensitivity.{pdf,png}` | Paper II assets |
| `ops/telemetry_job_resources` | MSI Agate Slurm FairShare accounting and `#SBATCH` rightsizing | Operations |

Design and model-selection rationale lives in [`docs/decisions.md`](../docs/decisions.md). Legacy per-suite notebooks remain runnable in `defunct/notebooks_by_suite/`.


In [1]:
# ---------------------------------------------------------------- settings
SNAPSHOT = "latest"      # "latest" | time stamp or prefix | "newest" | path/to/snapshot.zip | "live"
SAVE_FIGURES = False     # write PDF+PNG of each figure to figures/00/
# ---------------------------------------------------------------------------
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "knrates").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
for _m in [m for m in list(sys.modules) if m.split(".")[0] == "knrates"]:
    del sys.modules[_m]   # always import the repository copy (a kernel may predate a git pull)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from knrates import notebook as NB, figures as F, analysis as A, plotting as P, literature as L
from knrates.stats import summarize_runs

ctx = NB.setup(SNAPSHOT, save_figures=SAVE_FIGURES, fig_dir="figures/00", quiet=True)
runs, lim, M = ctx.runs, ctx.lim, ctx.models
FIX, ISO = "Bu2026Fixed", "Bu2026Vary"


## 1. What is in the simulation and archival-search snapshots


In [2]:
from knrates import load_snapshot
raw = load_snapshot(SNAPSHOT)      # unfiltered: every simulation suite and model
inv = []
for t, label in [("suite01_runs", "01 fixed-parameter ensemble"),
                 ("suite07_runs", "07 GW170817 parameter set"),
                 ("suite03_runs", "03 regimes x populations"),
                 ("suite02_candidate_runs", "02 criteria-sweep candidate sets"),
                 ("suite08_runs", "08 matched-exposure ensembles")]:
    d = raw.get(t)
    if d is None or d.empty:
        continue
    key = "population_type" if "population_type" in d else "model"
    for m, g in d.groupby(key):
        inv.append(dict(suite=label, model=m, realisations=len(g),
                        injected=g["n_simulated"].sum() if "n_simulated" in g else g.get("n_transients", pd.Series([np.nan])).sum()))
inv = pd.DataFrame(inv)
inv["used in Paper I baseline"] = inv.model.isin(M)
print("Simulation snapshot:", raw.label)
display(inv.style.format({"injected": "{:,.0f}"}).hide(axis="index"))

if ctx.search_snap is not None:
    print("\nArchival search snapshot:", ctx.search_snap.label)
    s_inv = pd.DataFrame([
        dict(table=name, rows=len(df), columns=len(df.columns))
        for name, df in sorted(ctx.search_snap.tables.items())
    ])
    display(s_inv.style.hide(axis="index"))
else:
    print("\nArchival search snapshot not present on disk (simulation-only mode).")


Simulation snapshot: knrates_results_latest.zip (created 20261111T024800Z, git d7af6653)


suite,model,realisations,injected,used in Paper I baseline
01 fixed-parameter ensemble,Bu2026Fixed,100,"100,000,000",True
01 fixed-parameter ensemble,Bu2026Vary,100,"100,000,000",True
01 fixed-parameter ensemble,Metzger,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Bu2026_FixedView,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Bu2026_Isotropic,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Metzger_Blue,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Metzger_Mean,100,"100,000,000",False
03 regimes x populations,AT2017gfo_FixedView,9,"9,000,000",False
03 regimes x populations,AT2017gfo_IsotropicView,9,"9,000,000",False
03 regimes x populations,BroadPopulation_Isotropic,9,"9,000,000",False



Archival search snapshot: knsearch_results_latest.zip (created 20261126T031508Z, git b4db2735)


table,rows,columns
coverage_boom_retention,103,8
coverage_irsa,239,24
golden_objects,20,11
rb_calibration,3,7
stage_a_cutflow,64,5
stage_a_runs,8,8


## 2. Headline findings (recomputed live from snapshots)


In [3]:
cmp = A.andreoni_comparison(ctx.tm, runs, M).set_index("model")
inc = A.inclination_efficiency(ctx.src, runs, ISO, n_bins=20)
xc = A.inclination_crosscheck(inc, lim, ISO, FIX)
dq = A.distance_quantiles(ctx.zh, FIX)
s_iso = ctx.src[ctx.src.model == ISO]
r07 = NB.all_limits(SNAPSHOT)
fid02 = ctx.snap["suite02_fiducial"].set_index("model")
resp = A.criteria_response(ctx.snap["suite02_slices_1d"], fid02)
top = resp[resp.model == FIX].sort_values("vt_max_over_min").iloc[-1]
s3 = ctx.snap["suite03_runs"].rename(columns={"population_type": "model"})
l3 = summarize_runs(s3, by=("criteria_regime", "model")).set_index(["criteria_regime", "model"])
a21 = L.lookup("andreoni2021_ztfrest_combined")
e = A.vt_by_epoch(ctx.tm, runs, FIX); e = e[e.vt_cumulative > 0]
upto = lambda d: e[e.month_end < pd.Timestamp(d) + pd.Timedelta(days=1)].iloc[-1].vt_cumulative
a21_pred = L.lookup("andreoni2020_gw170817")["limit"] * upto(L.lookup("andreoni2020_gw170817")["data_end"]) / upto(a21["data_end"])

claims = []
if ctx.search_snap is not None:
    sa = ctx.search_snap.get("stage_a_runs", pd.DataFrame())
    rb = ctx.search_snap.get("rb_calibration", pd.DataFrame())
    go = ctx.search_snap.get("golden_objects", pd.DataFrame())
    if len(sa):
        claims.append(("Stage A benchmark objects scanned / selected",
                       f"{sa.n_all_objects.sum():,.0f} / {sa.n_selected.sum():,.0f} ({sa.n_all_objects.sum() / max(sa.n_selected.sum(), 1):,.0f}x reduction)",
                       "01 · Paper I §3.2"))
    if len(rb):
        th_str = ", ".join(f"{r.model.split('_')[0]}={r.threshold:.3f}" for _, r in rb.iterrows())
        claims.append(("Calibrated RB fallback thresholds (matched to drb >= 0.5)", th_str, "01 · Paper I §3.2"))
    if len(go):
        g_pub = go[go.stream == "public"]
        g_all = go[go.stream == "public_partnership"]
        claims.append(("Golden fast transients passing full criteria (public / public+partnership)",
                       f"{int(g_pub.passes_criteria.sum())}/{len(g_pub)} vs. {int(g_all.passes_criteria.sum())}/{len(g_all)}",
                       "01 · Paper I §3.7"))

claims += [
    ("R95 baseline, AT2017gfo-like ejecta at iota = 25.8 deg [Gpc^-3 yr^-1]", f"{lim.loc[FIX, 'r95']:.0f}", "02 · Paper I §5.1, Table 1"),
    ("R95 baseline, same ejecta, isotropic viewing [Gpc^-3 yr^-1]", f"{lim.loc[ISO, 'r95']:.0f}", "02 · Paper I §5.1, Table 1"),
    ("⟨VT⟩ polar / ⟨VT⟩ isotropic", f"{lim.loc[FIX, 'vt_gpc3_yr'] / lim.loc[ISO, 'vt_gpc3_yr']:.2f}", "02, 03 · Paper I §5.3"),
    ("Polar baseline limit inside GWTC-4.0 BNS rate range (7.6-250)?", str(7.6 <= lim.loc[FIX, 'r95'] <= 250), "02 · Paper I Fig. 3"),
    ("Isotropic baseline limit inside GWTC-4.0 BNS rate range?", str(7.6 <= lim.loc[ISO, 'r95'] <= 250), "02 · Paper I Fig. 3"),
    ("Half / 99% of polar recoveries within d_L [Mpc]", f"{dq['dL50']:.0f} / {dq['dL99']:.0f}", "02 · Paper I Fig. 4"),
    ("MC uncertainty of pooled ⟨VT⟩ (polar / isotropic)", f"{100 * lim.loc[FIX, 'vt_err_empirical'] / lim.loc[FIX, 'vt_gpc3_yr']:.2f}% / {100 * lim.loc[ISO, 'vt_err_empirical'] / lim.loc[ISO, 'vt_gpc3_yr']:.2f}%", "04 · Paper I App. A"),
    ("R95 in Andreoni+2020's 23-month window (polar / iso)", f"{cmp.loc[FIX, 'r95_window']:.0f} / {cmp.loc[ISO, 'r95_window']:.0f}", "02 · Paper I §5.2"),
    ("... change vs. published 1775 / 4029", f"{100 * cmp.loc[FIX, 'window_change_vs_published']:+.0f}% / {100 * cmp.loc[ISO, 'window_change_vs_published']:+.0f}%", "02 · Paper I Fig. 1"),
    ("⟨VT⟩ growth, 8-yr survey vs. 23 months (duration alone)", f"{cmp.loc[FIX, 'vt_gain_full_over_window']:.2f}x ({cmp.loc[FIX, 'duration_gain']:.2f}x)", "02 · Paper I §5.2"),
    ("Andreoni+2020 limits carried to 8 yr (polar / iso)", f"{cmp.loc[FIX, 'published_projected_full']:.0f} / {cmp.loc[ISO, 'published_projected_full']:.0f}", "02 · Paper I §5.2"),
    ("ZTFReST 900 vs. A20 carried to its data end", f"900 vs. {a21_pred:.0f}", "02 · Paper I §5.2"),
    ("Isotropic: share of injections / recoveries at iota > 45 deg", f"{100 * np.cos(np.pi / 4):.0f}% / {100 * (s_iso.inclination_rad > np.pi / 4).mean():.0f}%", "03 · Paper I §5.3"),
    ("⟨VT⟩(iota): face-on bin / edge-on bin", f"{inc.sort_values('iota_deg_lo').vt.iloc[0] / inc.sort_values('iota_deg_lo').vt.iloc[-1]:.0f}x", "03 · Paper I Fig. 2"),
    ("Isotropic ⟨VT⟩(iota) interpolated at 25.8 deg vs. fixed-angle ensemble", f"{100 * xc['rel_diff']:+.2f}%", "03 · Paper I §5.3"),
]
if "GW170817_Bu2026_FixedView" in r07.index:
    claims.append(("Suite 07 (M_dyn -0.1 dex) ⟨VT⟩ change, polar / iso",
                   f"{100 * (r07.loc['GW170817_Bu2026_FixedView', 'vt_gpc3_yr'] / lim.loc[FIX, 'vt_gpc3_yr'] - 1):+.1f}% / "
                   f"{100 * (r07.loc['GW170817_Bu2026_Isotropic', 'vt_gpc3_yr'] / lim.loc[ISO, 'vt_gpc3_yr'] - 1):+.1f}%", "04 · Paper I §5.4"))
claims += [
    ("Suite 02 filter / Suite 01 at identical criteria (polar)", f"{fid02.loc[FIX, 'effective_vt_gpc3_yr'] / lim.loc[FIX, 'vt_gpc3_yr']:.2f}", "companion/05 · Paper II"),
    ("Criterion with largest ⟨VT⟩ dynamic range (polar)", f"{top.criterion}: {top.vt_max_over_min:.1f}x", "companion/05 · Paper II Fig. 6"),
]
if ("Baseline_ZTFReST", "BroadPopulation_Isotropic") in l3.index:
    claims.append(("Suite 03 broad population / AT2017gfo isotropic ⟨VT⟩ (baseline criteria)",
                   f"{l3.loc[('Baseline_ZTFReST', 'BroadPopulation_Isotropic'), 'vt_gpc3_yr'] / l3.loc[('Baseline_ZTFReST', 'AT2017gfo_IsotropicView'), 'vt_gpc3_yr']:.2f}", "04 · Paper I §5.4"))
pd.set_option("display.max_colwidth", 95)
display(pd.DataFrame(claims, columns=["claim", "value", "notebook · manuscript"]).style.hide(axis="index"))


claim,value,notebook · manuscript
Stage A benchmark objects scanned / selected,"9,725,527 / 2,703 (3,598x reduction)",01 · Paper I §3.2
Calibrated RB fallback thresholds (matched to drb >= 0.5),"t17=0.477, t15=0.293, t8=0.474",01 · Paper I §3.2
Golden fast transients passing full criteria (public / public+partnership),2/9 vs. 3/9,01 · Paper I §3.7
"R95 baseline, AT2017gfo-like ejecta at iota = 25.8 deg [Gpc^-3 yr^-1]",192,"02 · Paper I §5.1, Table 1"
"R95 baseline, same ejecta, isotropic viewing [Gpc^-3 yr^-1]",664,"02 · Paper I §5.1, Table 1"
⟨VT⟩ polar / ⟨VT⟩ isotropic,3.46,"02, 03 · Paper I §5.3"
Polar baseline limit inside GWTC-4.0 BNS rate range (7.6-250)?,True,02 · Paper I Fig. 3
Isotropic baseline limit inside GWTC-4.0 BNS rate range?,False,02 · Paper I Fig. 3
Half / 99% of polar recoveries within d_L [Mpc],143 / 275,02 · Paper I Fig. 4
MC uncertainty of pooled ⟨VT⟩ (polar / isotropic),0.23% / 0.45%,04 · Paper I App. A


## 3. Open `\tbd{...}` items in the manuscripts

Parsed live from `paper/ms.tex` and `paper/companion_paper_imposters_criteria.tex`.


In [4]:
import re
rows = []
for tex in ["paper/ms.tex", "paper/companion_paper_imposters_criteria.tex"]:
    text = (ROOT / tex).read_text()
    for mt in re.finditer(r"\\tbd\{", text):
        depth, i = 1, mt.end()
        while depth and i < len(text):
            depth += {"{": 1, "}": -1}.get(text[i], 0); i += 1
        line = text.count("\n", 0, mt.start()) + 1
        if text[text.rfind("\n", 0, mt.start()) + 1:mt.start()].lstrip().startswith("%"):
            continue
        body = " ".join(text[mt.end():i - 1].split())
        if body != "#1":
            rows.append(dict(file=tex.split("/")[-1], line=line, placeholder=body[:160] + ("…" if len(body) > 160 else "")))
display(pd.DataFrame(rows).style.hide(axis="index"))


file,line,placeholder
ms.tex,40,co-authors
ms.tex,43,Archival search result: $N_{\rm obs}$ surviving candidates and measured pipeline efficiency $\epspipe$.
ms.tex,43,"update with searched-exposure $\vteff \cdot \epspipe$, AT2017gfo posterior-marginalised limit, and numerical-relativity BNS population limit"
ms.tex,83,Insert Stage A--E sequential cutflow table from \texttt{snapshots/knsearch\_results\_latest.zip} once the full-survey 35-hour \texttt{ZTF\_alerts\_aux} scan (pr…
ms.tex,85,"Summarise empirical validation on golden fast optical transients in BOOM (e.g., GRB afterglow AT2021buv / ZTF21aagwbjr and fast optical transient AT2021any / ZT…"
ms.tex,87,Report surviving kilonova candidate count $N_{\rm obs}$ (and candidate dossiers if $N_{\rm obs} \ge 1$) together with the measured factorised pipeline efficienc…
ms.tex,94,software citation/DOI
ms.tex,119,Ensemble of posterior draws from published AT2017gfo POSSIS inference \citep{Anand2023} reporting posterior-marginalised $\vteff$ and 16th--84th percentile epis…
ms.tex,120,Per-merger population draws spanning dynamical and disk-wind ejecta priors motivated by BNS numerical-relativity simulations \citep{Radice2018} with isotropic v…
ms.tex,128,incorporate final searched-exposure $\vteff \cdot \epspipe$ and $N_{\rm obs}$
